# CETR-Net V2 — OFFICIAL FIVES 200-test evaluation, independent seed 2025
Locked one-time held-out test: 200 original records / 197 near-duplicate-screened groups.
Seven frozen full-data ablation checkpoints; primary hypothesis dual_full_asymmetric vs independent backbone. Threshold 0.5. Validation-only checkpoint choice. No test-derived model selection and no GPU laptop training.


In [ ]:

import os,sys,json,hashlib,runpy
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"
os.environ["CETR_OFFICIAL_SEED"]="2025"
import tensorflow as tf
gpus=tf.config.list_physical_devices("GPU")
print("OFFICIAL_TEST_ENV",{"seed":os.environ["CETR_OFFICIAL_SEED"],
                           "tf":tf.__version__,"gpu_count":len(gpus)},flush=True)
assert gpus,"GPU Kaggle required; laptop GPU prohibited and no CPU substitution"
work=Path("/kaggle/working")
p=list(Path("/kaggle/input").rglob("FROZEN_FIVES_V2_TEST_PLAN.json"))
assert len(p)==1
plan=json.loads(p[0].read_text(encoding="utf8"))
for fname in ("cetr_core_ablation.py","cetr_v2_suite.py"):
    matched=list(Path("/kaggle/input").rglob(fname))
    assert len(matched)==1,("Module missing or nonunique",fname)
    with matched[0].open("rb") as f: digest=hashlib.file_digest(f,"sha256").hexdigest()
    assert digest==plan["code_sha256"][fname],"Frozen code checksum changed"
    (work/fname).write_bytes(matched[0].read_bytes())
sys.path.insert(0,str(work))
print("OFFICIAL_FIVES_CODE_SOURCES_VERIFIED",flush=True)


In [ ]:
from pathlib import Path
import hashlib,runpy
work=Path("/kaggle/working")
payload='"""LOCKED FIVES V2 OFFICIAL held-out evaluation — Kaggle GPU, single seed per kernel.\nCompares frozen independent backbone vs seven predeclared refinement variants, with \nground truth read only after all prediction maps have been generated for each record.\nNO training, no validation selection, no test-time threshold fitting or GT-conditioned proposals.\n"""\nimport os,sys,datetime,hashlib,json,zipfile,time,platform,random\nfrom pathlib import Path\nfrom collections import defaultdict\nimport numpy as np,cv2,tensorflow as tf\nimport skimage\nfrom cetr_core_ablation import CETRNet,candidate\nfrom cetr_v2_suite import candidates_for_inference,scores,VARIANTS\n\nROOT=Path("/kaggle/working")\nOUT=ROOT/"CETR_OFFICIAL_TEST_V2"\nOUT.mkdir(parents=True,exist_ok=True)\nSEED=int(os.environ.get("CETR_OFFICIAL_SEED","42"))\nassert SEED in (42,123,2025)\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntf.random.set_seed(SEED)\ngpus=tf.config.list_physical_devices("GPU")\nassert gpus,"Kaggle GPU required for requested official evaluation; refusing CPU fallback"\nfor g in gpus:\n    try:tf.config.experimental.set_memory_growth(g,True)\n    except RuntimeError:pass\ndef current():return datetime.datetime.now(datetime.timezone.utc).isoformat()\ndef sha(path):\n    with Path(path).open("rb") as f:return hashlib.file_digest(f,"sha256").hexdigest()\ndef store(path,obj):\n    path=Path(path)\n    tmp=path.with_name(path.name+".tmp")\n    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False)+"\\n",encoding="utf8")\n    tmp.replace(path)\ndef lookup(name):\n    hits=list(Path("/kaggle/input").rglob(name))\n    assert len(hits)==1,("Expected one exact input file",name,[str(p) for p in hits])\n    return hits[0]\ntest_manifest=lookup("heldout_manifest.json")\ntest_base=test_manifest.parent\nmodel_manifest=lookup("SOURCE_MANIFEST.json")\nplan_path=lookup("FROZEN_FIVES_V2_TEST_PLAN.json")\nplan=json.loads(plan_path.read_text(encoding="utf8"))\nsource=json.loads(model_manifest.read_text(encoding="utf8"))\nassert plan["dataset_protocol"]=="FIVES_V2_NEARDUP_SCREENED"\nassert plan["threshold"]==.5 and plan["seeds"]==[42,123,2025]\nassert plan["test_records"]==200 and plan["unique_visual_groups"]==197\nassert plan["model_selection_source"].startswith("validation only")\nassert plan["ablation_reviewer_check_passed"] is True\nassert plan["primary_predeclared"]=="dual_full_asymmetric"\nassert plan["primary_comparator"]=="independent_backbone"\nassert len(plan["secondary_variants"])==7\nassert plan["inference_protocol"].startswith("prediction_only")\naudit=json.loads(lookup("HELDOUT_AUDIT.json").read_text(encoding="utf8"))\nassert audit["archive_sha256"]==plan["test_archive_sha256"],"Extracted held-out archive provenance mismatch"\nassert audit["records"]==plan["test_records"]==200 and audit["visual_groups"]==197\nhead_cache=lookup("seed42_natural_full_symmetric.npz").parent.parent\nassert head_cache.name=="heads_archive","Unexpected head checkpoint archive mount"\nassert len(list(head_cache.rglob("*.npz")))==21\nprint("FROZEN_HEADS_EXTRACTED_MOUNT_VERIFIED",len(list(head_cache.rglob("*.npz"))),flush=True)\nassert source["official_test_scored"] is False\nweights=model_manifest.parent/f"backbone_seed{SEED}.weights.h5"\nassert weights.exists() and sha(weights)==source["seeds"][str(SEED)]["sha256"]\nassert sha(weights)==plan["backbones_sha256"][str(SEED)]\nassert source["seeds"][str(SEED)]["epochs_logged"]==45\nfor source_name in ("cetr_core_ablation.py","cetr_v2_suite.py"):\n    path=Path(__file__).parent/source_name\n    assert sha(path)==plan["code_sha256"][source_name],("Inference source mismatch",source_name)\nVAR_ORDER=plan["secondary_variants"]\nassert VAR_ORDER==["natural_full_symmetric","break_only","false_only","dual_full_symmetric",\n                   "dual_geometry","dual_confidence","dual_full_asymmetric"]\nheads={}\nfor variant in VAR_ORDER:\n    ref=plan["variants"][str(SEED)][variant]\n    path=head_cache/ref["checkpoint_file"]\n    assert path.is_file()\n    assert sha(path)==ref["checkpoint_sha256"],("Frozen head mismatch",SEED,variant)\n    assert ref["epochs_completed"]==8\n    with np.load(path) as npz:\n        names=tuple(npz.files)\n        weights_by_kind={}\n        for family in ("shared","restore","reject"):\n            indexed=[]\n            for key in names:\n                if key.startswith(family+"_"):\n                    ix=int(key.split("_")[-1])\n                    indexed.append((ix,np.asarray(npz[key]).copy()))\n            indexed.sort()\n            assert indexed and [i for i,_ in indexed]==list(range(len(indexed)))\n            assert all(np.all(np.isfinite(v)) for _,v in indexed)\n            weights_by_kind[family]=[v for _,v in indexed]\n    heads[variant]=weights_by_kind\nprint("FIVES_TEST_MODEL_PROVENANCE_VALID",{"seed":SEED,"backbone_sha256":sha(weights),\n      "variants":VAR_ORDER,"test_sha256":audit["archive_sha256"]},flush=True)\nmodel=CETRNet(shape=(512,512,3))\nmodel.backbone.load_weights(str(weights))\nmodel.backbone.trainable=False\ndummy=tf.zeros((1,512,512,3),tf.float32)\nprobe=candidate((20,20),(32,32),"restore",0,(512,512))\nmodel.forward(dummy,[probe],np.zeros((512,512),np.float32),training=False,evidence_mode="full")\nHEAD_LAYERS={"shared":model.shared,"restore":model.restore,"reject":model.reject}\nfor variant in VAR_ORDER:\n    for family,layer in HEAD_LAYERS.items():\n        learned=heads[variant][family]\n        assert len(learned)==len(layer.get_weights()),(variant,family)\n        for old,new in zip(learned,layer.get_weights()):\n            assert old.shape==new.shape,(variant,family,old.shape,new.shape)\ndel dummy\nimg_dir=OUT/"qualitative_fixed_ids"\nimg_dir.mkdir(exist_ok=True)\nPER_RECORD=OUT/"per_record_progress.jsonl"\nif PER_RECORD.exists():raise RuntimeError("Official seed evaluation already started: refusing overwrite")\nstart=time.monotonic()\nmetric_keys=("dice","iou","cldice")\nif True:\n    records=json.loads(test_manifest.read_text(encoding="utf8"))\n    assert len(records)==200\n    assert len({r["record_key"] for r in records})==200\n    assert len({r["file_id"] for r in records})==200\n    assert len({r["near_group"] for r in records})==197\n    assert all(r["group_size"]>=1 for r in records)\n    fixed_ids={records[j]["record_key"] for j in (0,39,79,119,159,199)}\n    all_rows=[]\n    with PER_RECORD.open("a",encoding="utf8") as f:\n        for i,rec in enumerate(records,1):\n            fid=rec["file_id"]\n            img=cv2.imread(str(test_base/"images"/(fid+".png")),cv2.IMREAD_COLOR)\n            assert img is not None and img.shape[:2]==(512,512)\n            im_rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)\n            x=tf.convert_to_tensor(im_rgb.astype("float32")[None]/255.)\n            p0,_=model.backbone(x,training=False)\n            pp=np.asarray(p0.numpy()[0,:,:,0],dtype=np.float32)\n            binary,cands=candidates_for_inference(pp)\n            assert len(cands)<=12\n            predictions={}\n            for variant in VAR_ORDER:\n                for family,layer in HEAD_LAYERS.items():\n                    layer.set_weights(heads[variant][family])\n                mode=VARIANTS[variant]["evidence"]\n                p,_,_,_,_=model.forward(x,cands,binary,training=False,evidence_mode=mode)\n                predictions[variant]=np.asarray(p.numpy()[0,:,:,0],dtype=np.float32)\n                assert predictions[variant].shape==(512,512)\n                assert np.all(np.isfinite(predictions[variant]))\n            # Crucial: open test annotations ONLY AFTER all model predictions exist.\n            gt=cv2.imread(str(test_base/"masks"/(fid+".png")),cv2.IMREAD_GRAYSCALE)\n            assert gt is not None and gt.shape==(512,512)\n            truth=(gt>127).astype(np.float32)\n            assert set(np.unique(truth)).issubset({0.,1.})\n            bas=scores(truth,pp)\n            variant_scores={v:scores(truth,predictions[v]) for v in VAR_ORDER}\n            row={"record_key":rec["record_key"],"near_group":rec["near_group"],\n                 "file_id":fid,"seed":SEED,"n_candidates":len(cands),\n                 "backbone":bas,"variants":variant_scores}\n            all_rows.append(row)\n            f.write(json.dumps(row,ensure_ascii=False)+"\\n")\n            f.flush()\n            if rec["record_key"] in fixed_ids:\n                target=predictions["dual_full_asymmetric"]\n                panel=np.concatenate([\n                    im_rgb,\n                    cv2.cvtColor((truth*255).astype(np.uint8),cv2.COLOR_GRAY2RGB),\n                    cv2.cvtColor(((pp>.5)*255).astype(np.uint8),cv2.COLOR_GRAY2RGB),\n                    cv2.cvtColor(((target>.5)*255).astype(np.uint8),cv2.COLOR_GRAY2RGB)],axis=1)\n                out=img_dir/(f"{i:03d}_input_GT_backbone_proposed.png")\n                assert cv2.imwrite(str(out),cv2.cvtColor(panel,cv2.COLOR_RGB2BGR))\n            if i==1 or i%25==0 or i==200:\n                state={"status":"RUNNING","seed":SEED,"n_processed":i,"n_total":200,\n                       "elapsed_sec":round(time.monotonic()-start,2),"time_utc":current()}\n                store(OUT/"PROGRESS.json",state)\n                print("FIVES_OFFICIAL_TEST_PROGRESS",json.dumps(state),flush=True)\n                del state\nassert len(all_rows)==200\nvisual_groups=sorted({r["near_group"] for r in all_rows})\nassert len(visual_groups)==197\ngroupwise={}\nfor grp in visual_groups:\n    subgroup=[x for x in all_rows if x["near_group"]==grp]\n    groupwise[grp]={"n_records":len(subgroup),\n      "backbone":{k:float(np.mean([r["backbone"][k] for r in subgroup])) for k in metric_keys},\n      "variants":{v:{k:float(np.mean([r["variants"][v][k] for r in subgroup])) for k in metric_keys} for v in VAR_ORDER}}\nstore(OUT/"groupwise_197.json",groupwise)\nstore(OUT/"per_record_200.json",all_rows)\ndef ci_and_delta(a,b,seedcode):\n    a=np.asarray(a,np.float64)\n    b=np.asarray(b,np.float64)\n    assert len(a)==len(b)==197\n    rng=np.random.default_rng(seedcode)\n    draw=rng.integers(0,197,size=(2000,197))\n    differences=a-b\n    bs=differences[draw].mean(axis=1)\n    return {"paired_delta_mean":float(np.mean(differences)),\n            "paired_delta_95pct_cluster_bootstrap_ci":[float(z) for z in np.quantile(bs,[.025,.975])],\n            "n_visual_groups":197}\nby_model={"independent_backbone":{\n    k:float(np.mean([groupwise[g]["backbone"][k] for g in visual_groups])) for k in metric_keys}}\nbaseline_dice=[groupwise[g]["backbone"]["dice"] for g in visual_groups]\nvariant_results={}\nfor idx,v in enumerate(VAR_ORDER):\n    values=[groupwise[g]["variants"][v]["dice"] for g in visual_groups]\n    m={k:float(np.mean([groupwise[g]["variants"][v][k] for g in visual_groups])) for k in metric_keys}\n    m["vs_backbone"]=ci_and_delta(values,baseline_dice,SEED*100+idx)\n    m["vs_natural"]=ci_and_delta(values,[groupwise[g]["variants"]["natural_full_symmetric"]["dice"] for g in visual_groups],SEED*200+idx)\n    variant_results[v]=m\nsummary={\n    "status":"COMPLETE","seed":SEED,"source":"Kaggle GPU private official FIVES V2 held-out",\n    "completed_utc":current(),"n_records":200,"n_unique_visual_groups":197,\n    "n_variants":7,"threshold":plan["threshold"],"primary_variant":"dual_full_asymmetric",\n    "primary_comparator":"independent_backbone",\n    "prior_validation_best_variant":plan["validation_best_variants_descriptive_only"][str(SEED)]["variant"],\n    "backbone":by_model["independent_backbone"],"refiners":variant_results,\n    "test_archive_sha256":plan["test_archive_sha256"],\n    "backbone_sha256":plan["backbones_sha256"][str(SEED)],\n    "frozen_plan_sha256":sha(plan_path),"validation_only_model_selection":True,\n    "test_time_ground_truth_proposals":False,\n    "groupwise_statistics":"unweighted mean across 197 screened visual groups, paired 2000 bootstrap CI",\n    "limitations":["Near-duplicate visual grouping does not demonstrate patient-disjoint test",\n                   "2D FIVES masks only; no 3D anatomical connectivity validation",\n                   "No newly trained protocol-matched MaskVSC or SOTA baselines in this test",\n                   "Secondary 7-variant comparisons are descriptive; no test-time model selection"],\n    "runtime":{"tf":tf.__version__,"numpy":np.__version__,"skimage":skimage.__version__,\n               "kaggle_gpu_count":len(gpus),"elapsed_seconds":round(time.monotonic()-start,2)}}\nstore(OUT/"OFFICIAL_FIVES_V2_SEED_SUMMARY.json",summary)\nstore(OUT/"PROGRESS.json",{"status":"COMPLETE","seed":SEED,"n_processed":200,"n_total":200,"time_utc":current()})\nprint("FIVES_V2_OFFICIAL_TEST_SEED_COMPLETE",SEED,\n      "BASELINE_DICE",round(summary["backbone"]["dice"],6),\n      "PRIMARY_DICE",round(summary["refiners"]["dual_full_asymmetric"]["dice"],6),\n      "PRIMARY_DELTA",round(summary["refiners"]["dual_full_asymmetric"]["vs_backbone"]["paired_delta_mean"],6),\n      flush=True)\n'
code_file=work/"evaluate_official_fives_kaggle.py"
code_file.write_text(payload,encoding="utf8")
digest=hashlib.sha256(payload.encode("utf8")).hexdigest()
print("EVALUATOR_SCRIPT_SHA256",digest,flush=True)
runpy.run_path(str(code_file),run_name="__main__")
